# Rainfall vs. Discharge — Single-Year & Multi-Year Plots

**Purpose:** Generates flexible rainfall–discharge plots for individual years
or a user-defined multi-year window, useful for visually inspecting model
performance and precipitation–runoff dynamics.

**What it does:**
- Loads precipitation (`Loop_6_TimeSeries_ZR.csv`) and observed daily Q
- Produces single-year and multi-year (2015–2023) combined plots
- Includes a scatter-style deviation view and a reservoir storage panel

**Input:** `Loop_6_TimeSeries_ZR.csv`, `Dily Q_Obs.xlsx`,
           `Deviation_and_Reservoir.xlsx`  
**Output:** Rainfall vs. discharge plots (single-year and multi-year)

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
YEAR = 2023


precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
discharge_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Dily Q_Obs.xlsx"

# ======================================================
# STYLE
# ======================================================
plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12
})

COLOR_Q = "blue"   # discharge
COLOR_P = "green"   # rainfall

# ======================================================
# READ & PROCESS RAINFALL (HOURLY → DAILY)
# ======================================================
df_p = pd.read_csv(precip_path, sep=";")
df_p.columns = df_p.columns.str.strip()

df_p["datetime"] = pd.to_datetime(df_p["datetime"], dayfirst=True, errors="coerce")
df_p = df_p.dropna(subset=["datetime"])

df_p = (
    df_p.set_index("datetime")
    .resample("D")["precip"]
    .sum()
    .reset_index()
)

df_p = df_p[df_p["datetime"].dt.year == YEAR]

# ======================================================
# READ DISCHARGE (DAILY)
# ======================================================
df_q = pd.read_excel(discharge_path)
df_q.columns = df_q.columns.str.strip()

df_q["Zeit"] = pd.to_datetime(df_q["Zeit"], dayfirst=True, errors="coerce")
df_q = df_q.dropna(subset=["Zeit"])
df_q = df_q[df_q["Zeit"].dt.year == YEAR]

# ======================================================
# MERGE
# ======================================================
df = pd.merge(df_p, df_q, left_on="datetime", right_on="Zeit", how="inner")
df = df.sort_values("datetime")

if df.empty:
    raise ValueError(f"No data available for {YEAR}")

# ======================================================
# PLOT
# ======================================================
fig, ax1 = plt.subplots(figsize=(16, 7))

# ---- Discharge (LEFT AXIS) ----
q_max = df["Obs (Q)"].max()

ax1.plot(
    df["datetime"],
    df["Obs (Q)"],
    color=COLOR_Q,
    linewidth=2.2,
    label="Discharge (m³/s)"
)
ax1.set_ylabel("Discharge (m³/s)", color=COLOR_Q)
ax1.tick_params(axis="y", labelcolor=COLOR_Q)
ax1.set_ylim(0, q_max * 1.3)

# ---- Rainfall (RIGHT AXIS) ----
ax2 = ax1.twinx()
p_max = df["precip"].max()

ax2.bar(
    df["datetime"],
    df["precip"],
    width=0.9,
    color=COLOR_P,
    alpha=0.6,
    label="Rainfall (mm)"
)
ax2.set_ylabel("Daily Rainfall (mm)", color=COLOR_P)
ax2.tick_params(axis="y", labelcolor=COLOR_P)
ax2.set_ylim(0, p_max * 1.4)
ax2.invert_yaxis()

# ---- X AXIS (MONTHS) ----
ax1.set_xlim(df["datetime"].min(), df["datetime"].max())
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

# ---- GRID (ONLY ONE GRID – IMPORTANT) ----
ax1.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
ax1.grid(True, axis="x", linestyle=":", linewidth=0.5, alpha=0.4)
ax1.set_axisbelow(True)
ax2.grid(False)   

# ---- TITLE & LEGEND ----
plt.title(f"Daily Rainfall–Discharge Response ({YEAR})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
#ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ======================================================
# USER INPUT
# ======================================================
YEAR = 2017

precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
discharge_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Dily Q_Obs.xlsx"

# ======================================================
# STYLE
# ======================================================
plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12
})

COLOR_Q = "#1f77b4"   # discharge
COLOR_P = "#ff7f0e"   # rainfall

# ======================================================
# READ & PROCESS DAILY RAINFALL
# ======================================================
df_p = pd.read_csv(precip_path, sep=";")
df_p.columns = df_p.columns.str.strip()
df_p["datetime"] = pd.to_datetime(df_p["datetime"], dayfirst=True, errors="coerce")
df_p = df_p.dropna(subset=["datetime"])

df_p = (
    df_p.set_index("datetime")
    .resample("D")["precip"]
    .sum()
    .reset_index()
)

# ======================================================
# READ DAILY DISCHARGE
# ======================================================
df_q = pd.read_excel(discharge_path)
df_q.columns = df_q.columns.str.strip()
df_q["Zeit"] = pd.to_datetime(df_q["Zeit"], dayfirst=True, errors="coerce")
df_q = df_q.dropna(subset=["Zeit"])

# ======================================================
# MERGE
# ======================================================
df = pd.merge(df_p, df_q, left_on="datetime", right_on="Zeit", how="inner")
df = df.sort_values("datetime").reset_index(drop=True)

# ======================================================
# YEAR MASK
# ======================================================
df_year = df[df["datetime"].dt.year == YEAR].copy()
if df_year.empty:
    raise ValueError(f"No data found for year {YEAR}")

# ======================================================
# CUMULATIVE SUMS
# ======================================================
df_year["cum_rain_mm"] = df_year["precip"].cumsum()
df_year["cum_Q_m3"] = (df_year["Obs (Q)"] * 86400).cumsum()  # daily discharge to m³

# ======================================================
# PLOT
# ======================================================
fig, ax1 = plt.subplots(figsize=(16, 7))

# ---- Cumulative Discharge (LEFT Y) ----
q_max = df_year["cum_Q_m3"].max()
ax1.plot(
    df_year["datetime"],
    df_year["cum_Q_m3"],
    color=COLOR_Q,
    linewidth=2.2,
    label="Cumulative Discharge (m³)"
)
ax1.set_ylabel("Cumulative Discharge (m³)", color=COLOR_Q)
ax1.tick_params(axis="y", labelcolor=COLOR_Q)
ax1.set_ylim(0, q_max * 1.05)

# ---- Cumulative Rainfall (RIGHT Y) ----
p_max = df_year["cum_rain_mm"].max()
ax2 = ax1.twinx()
ax2.plot(
    df_year["datetime"],
    df_year["cum_rain_mm"],
    color=COLOR_P,
    linewidth=2.0,
    linestyle="--",
    label="Cumulative Rainfall (mm)"
)
ax2.set_ylabel("Cumulative Rainfall (mm)", color=COLOR_P)
ax2.tick_params(axis="y", labelcolor=COLOR_P)
ax2.set_ylim(0, p_max * 1.05)
ax2.grid(False)  # prevent double grid

# ---- X-axis formatting (months) ----
ax1.set_xlim(df_year["datetime"].min(), df_year["datetime"].max())
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

# ---- GRID, TITLE, LEGEND ----
ax1.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
ax1.grid(True, axis="x", linestyle=":", linewidth=0.5, alpha=0.4)

plt.title(f"Cumulative Rainfall and Discharge ({YEAR})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

# ======================================================
# PRINT NUMERICAL SUMMARY
# ======================================================
total_rain_mm = df_year["cum_rain_mm"].iloc[-1]
total_Q_m3 = df_year["cum_Q_m3"].iloc[-1]
mean_Q = df_year["Obs (Q)"].mean()

print(f"\n=== Cumulative Summary for {YEAR} ===")
print(f"Total Rainfall: {total_rain_mm:.1f} mm")
print(f"Total Discharge Volume: {total_Q_m3:,.0f} m³")
print(f"Mean Daily Discharge: {mean_Q:.3f} m³/s")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ======================================================
# USER INPUT
# ======================================================
START_YEAR = 2015
END_YEAR = 2023

precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
discharge_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Dily Q_Obs.xlsx"

# ======================================================
# STYLE
# ======================================================
plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12
})

COLOR_Q = "#1f77b4"   # discharge
COLOR_P = "#ff7f0e"   # rainfall

# ======================================================
# READ & PROCESS DAILY RAINFALL
# ======================================================
df_p = pd.read_csv(precip_path, sep=";")
df_p.columns = df_p.columns.str.strip()
df_p["datetime"] = pd.to_datetime(df_p["datetime"], dayfirst=True, errors="coerce")
df_p = df_p.dropna(subset=["datetime"])

df_p = (
    df_p.set_index("datetime")
    .resample("D")["precip"]
    .sum()
    .reset_index()
)

# ======================================================
# READ DAILY DISCHARGE
# ======================================================
df_q = pd.read_excel(discharge_path)
df_q.columns = df_q.columns.str.strip()
df_q["Zeit"] = pd.to_datetime(df_q["Zeit"], dayfirst=True, errors="coerce")
df_q = df_q.dropna(subset=["Zeit"])

# ======================================================
# MERGE
# ======================================================
df = pd.merge(df_p, df_q, left_on="datetime", right_on="Zeit", how="inner")
df = df.sort_values("datetime").reset_index(drop=True)

# ======================================================
# YEAR RANGE MASK
# ======================================================
df_range = df[(df["datetime"].dt.year >= START_YEAR) & (df["datetime"].dt.year <= END_YEAR)].copy()
if df_range.empty:
    raise ValueError(f"No data found between {START_YEAR} and {END_YEAR}")

# ======================================================
# CUMULATIVE SUMS
# ======================================================
df_range["cum_rain_mm"] = df_range["precip"].cumsum()
df_range["cum_Q_m3"] = (df_range["Obs (Q)"] * 86400).cumsum()  # daily discharge → m³

# ======================================================
# PLOT CUMULATIVE RAINFALL & DISCHARGE
# ======================================================
fig, ax1 = plt.subplots(figsize=(16, 7))

# ---- Cumulative Discharge (LEFT Y) ----
q_max = df_range["cum_Q_m3"].max()
ax1.plot(
    df_range["datetime"],
    df_range["cum_Q_m3"],
    color=COLOR_Q,
    linewidth=2.2,
    label="Cumulative Discharge (m³)"
)
ax1.set_ylabel("Cumulative Discharge (m³)", color=COLOR_Q)
ax1.tick_params(axis="y", labelcolor=COLOR_Q)
ax1.set_ylim(0, q_max * 1.05)

# ---- Cumulative Rainfall (RIGHT Y) ----
p_max = df_range["cum_rain_mm"].max()
ax2 = ax1.twinx()
ax2.plot(
    df_range["datetime"],
    df_range["cum_rain_mm"],
    color=COLOR_P,
    linewidth=2.0,
    linestyle="--",
    label="Cumulative Rainfall (mm)"
)
ax2.set_ylabel("Cumulative Rainfall (mm)", color=COLOR_P)
ax2.tick_params(axis="y", labelcolor=COLOR_P)
ax2.set_ylim(0, p_max * 1.05)
ax2.grid(False)  # prevent double grid

# ---- X-axis formatting (years/months) ----
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.set_xlim(df_range["datetime"].min(), df_range["datetime"].max())

# ---- GRID, TITLE, LEGEND ----
ax1.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
ax1.grid(True, axis="x", linestyle=":", linewidth=0.5, alpha=0.4)

plt.title(f"Cumulative Rainfall and Discharge ({START_YEAR}-{END_YEAR})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

# ======================================================
# PRINT NUMERICAL SUMMARY
# ======================================================
total_rain_mm = df_range["cum_rain_mm"].iloc[-1]
total_Q_m3 = df_range["cum_Q_m3"].iloc[-1]
mean_Q = df_range["Obs (Q)"].mean()

print(f"\n=== Cumulative Summary ({START_YEAR}-{END_YEAR}) ===")
print(f"Total Rainfall: {total_rain_mm:.1f} mm")
print(f"Total Discharge Volume: {total_Q_m3:,.0f} m³")
print(f"Mean Daily Discharge: {mean_Q:.3f} m³/s")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ======================
# USER INPUT
# ======================
START_YEAR = 2015
END_YEAR = 2023

precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
discharge_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Dily Q_Obs.xlsx"


# ======================
# READ DATA
# ======================
# Rainfall
df_p = pd.read_csv(precip_path, sep=";")
df_p.columns = df_p.columns.str.strip()
df_p["datetime"] = pd.to_datetime(df_p["datetime"], dayfirst=True, errors="coerce")
df_p = df_p.dropna(subset=["datetime"])
df_p = df_p.set_index("datetime").resample("D")["precip"].sum().reset_index()

# Discharge
df_q = pd.read_excel(discharge_path)
df_q.columns = df_q.columns.str.strip()
df_q["Zeit"] = pd.to_datetime(df_q["Zeit"], dayfirst=True, errors="coerce")
df_q = df_q.dropna(subset=["Zeit"])

# Merge
df = pd.merge(df_p, df_q, left_on="datetime", right_on="Zeit", how="inner")
df = df.sort_values("datetime").reset_index(drop=True)

# Filter year range
df = df[(df["datetime"].dt.year >= START_YEAR) & (df["datetime"].dt.year <= END_YEAR)]
if df.empty:
    raise ValueError(f"No data found between {START_YEAR} and {END_YEAR}")
# ======================
# PLOT
# ======================
fig, ax1 = plt.subplots(figsize=(16,6))

COLOR_Q = "#1f77b4"   # blue for discharge
COLOR_P = "#ff7f0e"   # orange for rainfall

# ---- Daily Discharge (line) ----
ax1.plot(df["datetime"], df["Obs (Q)"], color=COLOR_Q, linewidth=2.0, label="Daily Discharge (m³/s)")
ax1.set_ylabel("Discharge (m³/s)", color=COLOR_Q)
ax1.tick_params(axis="y", labelcolor=COLOR_Q)
ax1.set_ylim(0, df["Obs (Q)"].max()*1.3)

# ---- Daily Rainfall (bars) ----
ax2 = ax1.twinx()
ax2.bar(df["datetime"], df["precip"], width=1.0, color=COLOR_P, alpha=0.5, edgecolor="#c65100", linewidth=0.7, label="Daily Rainfall (mm)")
ax2.set_ylabel("Daily Rainfall (mm)", color=COLOR_P)
ax2.tick_params(axis="y", labelcolor=COLOR_P)
ax2.set_ylim(0, df["precip"].max()*1.4)
ax2.invert_yaxis()  # hydrological convention

# ---- X-axis formatting ----
import matplotlib.dates as mdates
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.set_xlim(df["datetime"].min(), df["datetime"].max())
ax1.grid(True, linestyle="--", alpha=0.6)

# ---- LEGEND & TITLE ----
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
#ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", frameon=True)

plt.title(f"Daily Rainfall vs Discharge ({START_YEAR}-{END_YEAR})", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ======================
# USER INPUT
# ======================
START_YEAR = 2015
END_YEAR = 2023
data_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Deviation_and_Reservoir.xlsx"
precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"

# ======================
# READ DAILY RAINFALL
# ======================
df_rain = pd.read_csv(precip_path, sep=";")
df_rain.columns = df_rain.columns.str.strip()
df_rain["datetime"] = pd.to_datetime(df_rain["datetime"], dayfirst=True, errors="coerce")
df_rain = df_rain.dropna(subset=["datetime"])
df_rain = df_rain.set_index("datetime").resample("D")["precip"].sum().reset_index()

# Filter year range
df_rain = df_rain[(df_rain["datetime"].dt.year >= START_YEAR) & (df_rain["datetime"].dt.year <= END_YEAR)]

# ======================
# READ OBSERVED & SIMULATED DISCHARGE
# ======================
df_q = pd.read_excel(data_path)
df_q.columns = df_q.columns.str.strip()
df_q["Zeit"] = pd.to_datetime(df_q["Zeit"], dayfirst=True, errors="coerce")
df_q = df_q.dropna(subset=["Zeit"])
df_q = df_q[(df_q["Zeit"].dt.year >= START_YEAR) & (df_q["Zeit"].dt.year <= END_YEAR)]

# Convert daily m3 to m3/s
df_q["Obs_Daily_m3s"] = df_q["Obs_Daily_m3"] / 86400
df_q["Sim_Daily_m3s"] = df_q["Sim_Daily_m3"] / 86400

# ======================
# MERGE RAINFALL WITH DISCHARGE
# ======================
df = pd.merge(df_rain, df_q, left_on="datetime", right_on="Zeit", how="inner")
df = df.sort_values("datetime").reset_index(drop=True)

# ======================
# PLOT MULTI-PANEL
# ======================
fig, axes = plt.subplots(2, 1, figsize=(18,16), sharex=True)

COLOR_Q_OBS = "#1f77b4"
COLOR_Q_SIM = "#2ca02c"
COLOR_P = "#ff7f0e"

# --- Panel 1: Rainfall vs Observed Discharge ---
ax1 = axes[0]
ax1.plot(df["datetime"], df["Obs_Daily_m3s"], color=COLOR_Q_OBS, linewidth=2, label="Observed Discharge (m³/s)")
ax2 = ax1.twinx()
ax2.bar(df["datetime"], df["precip"], width=1.0, color=COLOR_P, alpha=0.6, label="Daily Rainfall (mm)")
ax1.set_ylabel("Discharge (m³/s)", color=COLOR_Q_OBS)
ax2.set_ylabel("Daily Rainfall (mm)", color=COLOR_P)
ax1.tick_params(axis="y", labelcolor=COLOR_Q_OBS)
ax2.tick_params(axis="y", labelcolor=COLOR_P)
ax2.invert_yaxis()  # hyetograph style
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.set_title("Daily Rainfall vs Observed Discharge (2015–2023)")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# --- Panel 2: Rainfall vs Simulated Discharge ---
ax3 = axes[1]
ax3.plot(df["datetime"], df["Sim_Daily_m3s"], color=COLOR_Q_SIM, linewidth=2, label="Simulated Discharge (m³/s)")
ax4 = ax3.twinx()
ax4.bar(df["datetime"], df["precip"], width=1.0, color=COLOR_P, alpha=0.6, label="Daily Rainfall (mm)")
ax3.set_ylabel("Discharge (m³/s)", color=COLOR_Q_SIM)
ax4.set_ylabel("Daily Rainfall (mm)", color=COLOR_P)
ax3.tick_params(axis="y", labelcolor=COLOR_Q_SIM)
ax4.tick_params(axis="y", labelcolor=COLOR_P)
ax4.invert_yaxis()
ax3.grid(True, linestyle="--", alpha=0.6)
ax3.set_title("Daily Rainfall vs Simulated Discharge (2015–2023)")
lines3, labels3 = ax3.get_legend_handles_labels()
lines4, labels4 = ax4.get_legend_handles_labels()
ax3.legend(lines3 + lines4, labels3 + labels4, loc="upper left")

# --- X-axis formatting ---
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(df["datetime"].min(), df["datetime"].max())

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ======================
# HYDROLOGICAL METRICS
# ======================
def nse(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]
    if len(q_obs) == 0 or np.all(q_obs == q_obs[0]):
        return np.nan
    numerator = np.sum((q_obs - q_sim) ** 2)
    denominator = np.sum((q_obs - np.mean(q_obs)) ** 2)
    return 1 - (numerator / denominator)

def rmse(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]
    if len(q_obs) == 0:
        return np.nan
    return np.sqrt(np.mean((q_obs - q_sim) ** 2))

def kge(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]
    if len(q_obs) == 0:
        return np.nan, np.nan, np.nan, np.nan
    # Correlation
    r = np.corrcoef(q_obs, q_sim)[0,1] if np.std(q_obs) and np.std(q_sim) else np.nan
    mean_obs, mean_sim = np.mean(q_obs), np.mean(q_sim)
    std_obs, std_sim = np.std(q_obs), np.std(q_sim)
    cv_obs = std_obs / mean_obs if mean_obs != 0 else np.nan
    cv_sim = std_sim / mean_sim if mean_sim != 0 else np.nan
    beta = mean_sim / mean_obs if mean_obs != 0 else np.nan
    gamma = cv_sim / cv_obs if cv_obs != 0 else np.nan
    if np.isnan(r) or np.isnan(beta) or np.isnan(gamma):
        return np.nan, r, beta, gamma
    kge_value = 1 - np.sqrt((r-1)**2 + (gamma-1)**2 + (beta-1)**2)
    return kge_value, r, beta, gamma

# ======================
# MAIN
# ======================
def main():
    # Load Excel data
    file_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\adjusted_Calibration.xlsx"
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.strip()

    # Convert date and drop missing
    df['DateTime'] = pd.to_datetime(df['Zeit'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['DateTime', 'Obs', 'Sim'])

    # Convert from m³/day → m³/s
    df['Obs_m3s'] = df['Obs'] / 86400
    df['Sim_m3s'] = df['Sim'] / 86400

    # Print date range
    print("Data range:", df['DateTime'].min(), "to", df['DateTime'].max())

    # Calculate metrics
    NSE = nse(df['Obs_m3s'].values, df['Sim_m3s'].values)
    RMSE = rmse(df['Obs_m3s'].values, df['Sim_m3s'].values)
    KGE, r, beta, gamma = kge(df['Obs_m3s'].values, df['Sim_m3s'].values)

    # Hydrograph plot
    plt.figure(figsize=(14,7))
    plt.plot(df['DateTime'], df['Obs_m3s'], color='black', label='Observed', linewidth=1.5)
    plt.plot(df['DateTime'], df['Sim_m3s'], color='blue', linestyle='--', label='Simulated', linewidth=1.2)

    # Metrics text box
    textstr = (f"NSE   = {NSE:.3f}\n"
               f"RMSE  = {RMSE:.3f}\n"
               f"KGE   = {KGE:.3f}\n"
               f"r     = {r:.3f}\n"
               f"β     = {beta:.3f}\n"
               f"γ     = {gamma:.3f}")
    props = dict(boxstyle="round", facecolor="white", alpha=0.8)
    plt.text(0.02, 0.95, textstr, transform=plt.gca().transAxes,
             fontsize=12, verticalalignment='top', bbox=props)

    # X-axis formatting
    plt.xlabel("Date")
    plt.ylabel("Discharge (m³/s)")
    plt.title("Observed vs Simulated Discharge")
    plt.xlim(df['DateTime'].min(), df['DateTime'].max())
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()
